# Topic: SQL | GROUP BY + COUNT

## Definition (30-second explanation)
* `GROUP BY` + `COUNT` is a fundamental SQL aggregation pattern that collapses rows sharing the same value(s) in specified column(s) into a single output row, and counts how many original rows fall into each group.

## Why Interviewers Ask This
* It forms the backbone of almost every reporting and analytics query.
* It effectively tests your understanding of SQL's order of execution, specifically the difference between filtering raw data versus filtering aggregated data.
* It verifies you can distinguish between counting events (rows) versus counting unique entities (distinct values).

## Core Concepts
* **`GROUP BY`**: Collapses rows with the same value into one output row.
* **`COUNT(*)`**: Counts every single row in the group, including rows with `NULL` values.
* **`COUNT(column_name)`**: Counts only the rows where the specified column is NOT `NULL`.
* **`COUNT(DISTINCT column_name)`**: Counts only the unique, non-null values within the group.

## When to Use
* Whenever a business question asks "How many X per Y?" (e.g., logins per device, orders per product, users per country).
* When you need to quantify the volume of records associated with specific categories.

## Common Comparisons
* **`WHERE` vs. `HAVING`**: `WHERE` filters individual rows *before* aggregation; `HAVING` filters the grouped results *after* aggregation.
* **`COUNT(*)` vs. `COUNT(DISTINCT)`**: Use `COUNT(*)` for total volume/events; use `COUNT(DISTINCT)` when tracking unique entities (like unique customers instead of total orders).

## Common Interview Traps
* **Missing `GROUP BY` columns**: Forgetting to include all non-aggregated columns from the `SELECT` clause in the `GROUP BY` clause.
* **`WHERE` on Aggregates**: Using `WHERE COUNT(*) > 5` instead of `HAVING COUNT(*) > 5`.
* **Missing Aliases**: Failing to alias the calculated `COUNT` column, making the output unprofessional or hard to read.
* **Wrong Granularity**: Applying the `GROUP BY` on the wrong column relative to what the question asked for.

## SQL Syntax 
SELECT 
    category_column, 
    COUNT(*) AS total_count,                   -- Counts all rows
    COUNT(DISTINCT user_id) AS unique_users    -- Counts unique entities
FROM table_name
WHERE date >= '2025-01-01'                     -- Filter BEFORE grouping
GROUP BY category_column
HAVING COUNT(*) > 10                           -- Filter AFTER grouping
ORDER BY total_count DESC;                     -- Sort highest first

## 45-Second Interview Answer
"To find the frequency of records per category, I use `GROUP BY` alongside the `COUNT` function. If I need total rows, I'll use `COUNT(*)`. If I need to count unique entities and ignore duplicates, I'll use `COUNT(DISTINCT column)`. I also ensure that any row-level filtering is done in the `WHERE` clause before grouping, while any filtering on the aggregated counts is handled by the `HAVING` clause. Finally, I always alias my aggregates and use `ORDER BY` to format the final output clearly."

# Example Questions & Answers

**Q1. Count the number of products sold per product category.**
*   **Ideal Answer**: `SELECT product_category, COUNT(*) AS products_sold FROM sales GROUP BY product_category;`
*   **Common Mistake**: Forgetting to add an alias to `COUNT(*)`, leaving the column name as a raw function which is bad practice.
*   **Likely Follow-up**: How would you modify this to only show categories that sold more than 50 products? *(Answer: Add `HAVING COUNT(*) > 50` at the end).*

**Q2. Find the number of employees hired per year from the employees table.**
*   **Ideal Answer**: `SELECT EXTRACT(YEAR FROM hire_date) AS hire_year, COUNT(*) AS employees_hired FROM employees GROUP BY EXTRACT(YEAR FROM hire_date);` 
*   **Common Mistake**: Grouping by the exact `hire_date` timestamp instead of extracting and grouping by just the year.
*   **Likely Follow-up**: What if I want to know how many *unique roles* were hired per year? *(Answer: Change to `COUNT(DISTINCT role_id)`).*

**Q3. Count the number of support tickets per status (open, closed, pending).**
*   **Ideal Answer**: `SELECT status, COUNT(*) AS ticket_count FROM support_tickets GROUP BY status ORDER BY ticket_count DESC;`
*   **Common Mistake**: Not clarifying if they should use `COUNT(ticket_id)` or `COUNT(*)`.
*   **Likely Follow-up**: How do you ensure the status with the most tickets appears at the top? *(Answer: Use `ORDER BY ticket_count DESC`).*

**Q4. Find all countries that have more than 100 orders. (Hint: use HAVING)**
*   **Ideal Answer**: `SELECT country, COUNT(*) AS total_orders FROM orders GROUP BY country HAVING COUNT(*) > 100;`
*   **Common Mistake**: Using the `WHERE` clause (`WHERE COUNT(*) > 100`) to filter aggregated data.
*   **Likely Follow-up**: Can you filter by a specific year and also keep the `HAVING` clause? *(Answer: Yes, use `WHERE EXTRACT(YEAR FROM order_date) = 2025` before the `GROUP BY`).*

**Q5. Count the number of unique customers who placed at least one order per month.**
*   **Ideal Answer**: `SELECT DATE_TRUNC('month', order_date) AS order_month, COUNT(DISTINCT customer_id) AS unique_customers FROM orders GROUP BY DATE_TRUNC('month', order_date);`
*   **Common Mistake**: Confusing `COUNT(*)` (which counts total orders) with `COUNT(DISTINCT customer_id)` (which uniquely counts the customers).
*   **Likely Follow-up**: What happens to NULL `customer_id` values in `COUNT(DISTINCT customer_id)`? *(Answer: They are ignored/not counted).*

### Practice Questions:

In [1]:
import sqlite3
import pandas as pd

conn= sqlite3.connect('/home/shail/interview-prep/01_SQL/oracle_hr.db')

#### Q1. Multi-Column Grouping & Distinct Counts
**Question:** Write an SQL query to find the number of unique job roles (`job_id`) present in each department (`department_id`). Return only the departments that have more than 2 unique job roles, and order the final results so the department with the most unique roles appears first.

In [31]:
pd.read_sql_query(sql= """
SELECT 
    department_id, 
    COUNT(DISTINCT job_id) AS total_roles
FROM employees
GROUP BY department_id
HAVING COUNT(DISTINCT job_id) > 2
ORDER BY total_roles DESC;
""", con= conn)

,department_id,total_roles
0,50,3


#### Q2. Handling NULLs with LEFT JOIN + COUNT
**Question:** Write an SQL query to find the total number of employees in every department. Output the `department_name` and the employee count. You must include departments that currently have zero employees.

In [32]:
pd.read_sql_query(sql= """
SELECT 
    d.department_name, 
    COUNT(e.employee_id) AS num_emps
FROM departments d 
LEFT JOIN employees e
    ON d.department_id = e.department_id
GROUP BY 
    d.department_id, 
    d.department_name;
""", con= conn)

,department_name,num_emps
0,Administration,1
1,Marketing,2
2,Purchasing,6
3,Human Resources,1
4,Shipping,45
5,IT,5
6,Public Relations,1
7,Sales,34
8,Executive,3
9,Finance,6


#### Q3. Conditional Counting (Pivoting)
**Question:** Write an SQL query to find the total number of employees in each department, but break the count down into two separate columns: `high_earners` (salary > 10000) and `regular_earners` (salary <= 10000).

In [33]:
pd.read_sql_query(sql= """
SELECT
    department_id,
    SUM(CASE WHEN salary > 10000 THEN 1 ELSE 0 END) AS high_earners,
    SUM(CASE WHEN salary <= 10000 THEN 1 ELSE 0 END) AS regular_earners
FROM employees
GROUP BY department_id
ORDER BY department_id;
""", con= conn)

,department_id,high_earners,regular_earners
0,NaN,0,1
1,10.0,0,1
2,20.0,1,1
3,30.0,1,5
4,40.0,0,1
5,50.0,0,45
6,60.0,0,5
7,70.0,0,1
8,80.0,8,26
9,90.0,3,0
